# Error analysis

Which messages does the classifier get wrong, and is there a pattern?

Run from the repository root after `build_dataset.py` and `train.py`.
Everything here uses the same group-aware split as `evaluate.py`, so the
errors below are on genuinely unseen seeds.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from train import load_corpus, group_split, build_pipeline

df = load_corpus()
train, test = group_split(df)
pipe = build_pipeline("logreg")
pipe.fit(train["text"], train["label"])

scam_i = list(pipe.classes_).index("scam")
test = test.assign(
    pred=pipe.predict(test["text"]),
    p_scam=pipe.predict_proba(test["text"])[:, scam_i],
)
print(f"{len(test)} test rows from {test.seed_id.nunique()} unseen seeds")
print(f"errors: {(test.pred != test.label).sum()}")

## Where the errors concentrate

Errors cluster by *seed*, not by row: if a seed is misread, most of its ~16
variants are misread too. Counting rows overstates how many distinct mistakes
the model actually makes.

In [ ]:
errs = test[test.pred != test.label]
print("by language / true label:")
print(errs.groupby(["lang", "label"]).size())
print("\nby category:")
print(errs.groupby(["label", "category"]).size().sort_values(ascending=False))
print(f"\ndistinct seeds involved: {errs.seed_id.nunique()} "
      f"of {test.seed_id.nunique()} test seeds")
print(errs.seed_id.value_counts().head(10))

## False negatives — scams that got through

The expensive error. Sorted by confidence, so the most confidently-wrong
cases come first: those are the seeds whose wording the model has no
purchase on at all.

In [ ]:
fn = errs[errs.label == "scam"].sort_values("p_scam")
for _, r in fn.drop_duplicates("seed_id").head(12).iterrows():
    print(f"[{r.lang}/{r.category}] p_scam={r.p_scam:.3f}  {r.seed_id}")
    print(f"   {r.text}\n")

## False positives — legitimate messages flagged

The error that destroys trust. A tool that flags a real OTP or a real
operator promo gets uninstalled.

In [ ]:
fp = errs[errs.label == "ham"].sort_values("p_scam", ascending=False)
for _, r in fp.drop_duplicates("seed_id").head(12).iterrows():
    print(f"[{r.lang}] p_scam={r.p_scam:.3f}  {r.seed_id}")
    print(f"   {r.text}\n")

## Per-message explanations for the worst errors

`predict.py` returns the signed contribution of every feature, so a wrong
answer can be traced to the signal that caused it. This is how a bad lexicon
entry gets found.

In [ ]:
from predict import predict, describe

worst = pd.concat([fn.drop_duplicates("seed_id").head(3),
                   fp.drop_duplicates("seed_id").head(3)])
for _, r in worst.iterrows():
    res = predict(r.text, top_k=6)
    print(f"true={r.label}  pred={res.label}  p_scam={res.probability:.3f}")
    print(f"  {r.text}")
    for c in res.contributions:
        if c.kind == "signal":
            print(f"    {c.contribution:+.3f}  {describe(c)}")
    print()

## Threshold sweep

The 0.5 default is arbitrary. For a consumer warning tool a false positive is
cheaper than a missed scam, so a lower threshold may be the right operating
point — but it should be chosen deliberately, and on collected data rather
than on this authored corpus.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

y = (test.label == "scam").to_numpy()
print(f"{'thr':>5} {'prec':>6} {'recall':>7} {'F1':>6}")
for thr in np.arange(0.2, 0.85, 0.05):
    pred = np.where(test.p_scam >= thr, "scam", "ham")
    p, r, f, _ = precision_recall_fscore_support(
        test.label, pred, labels=["scam"], zero_division=0)
    print(f"{thr:5.2f} {p[0]:6.3f} {r[0]:7.3f} {f[0]:6.3f}")